# Qiskit Fall Fest 2026 — Challenge 2: Make the Quantum Coin Biased
**IBM Quantum | Vishnu Institute of Technology**  
**Name:** Srikar

## Objective
Build single-qubit circuits (max 4 gates before measurement, 100 shots) whose outcomes for `0` and `1` are **not** equally likely, predict the probabilities first, then compare with the measured results. Bonus: one circuit where `0` is more likely and one where `1` is more likely.

## 1. Setup

In [ ]:
%pip install -q qiskit qiskit-aer matplotlib pylatexenc

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

SHOTS = 100
simulator = AerSimulator()

def run(qc, shots=SHOTS):
    """Execute a circuit on the Aer simulator and return the counts."""
    return simulator.run(transpile(qc, simulator), shots=shots).result().get_counts()

def report(name, counts):
    z, o = counts.get('0', 0), counts.get('1', 0)
    print(f'{name}: 0 -> {z} ({z/SHOTS:.0%}), 1 -> {o} ({o/SHOTS:.0%})')

print('Setup complete. Shots per experiment:', SHOTS)

## 2. Idea and predictions

A single `H` gate gives a fair coin (50/50). To bias it we use a **phase** gate between two `H` gates, or rotate by a chosen angle.

| Circuit | Gates (before measurement) | Predicted P(0) | Predicted P(1) | Reason |
|---|---|---|---|---|
| Main (0-biased) | `H`, `T`, `H` | ≈ 85% | ≈ 15% | `T` adds a π/4 phase, so the second `H` no longer cancels perfectly: P(0) = cos²(π/8) ≈ 0.854 |
| Bonus (1-biased) | `X`, `H`, `T`, `H` | ≈ 15% | ≈ 85% | Starting from \|1⟩ flips the bias: P(0) = sin²(π/8) ≈ 0.146 |
| Degrees of bias | `RY(θ)` | cos²(θ/2) | sin²(θ/2) | `RY(θ)` rotates the qubit by angle θ about the Y axis |

**Think before you run:**
- `0` should occur more often in the main circuit (about 85 out of 100).
- Changing the gates (or the rotation angle) changes the percentages.
- With only 100 shots the counts will vary slightly from run to run.

## 3. Main circuit — `H`, `T`, `H` (0 is more likely)

In [ ]:
qc_main = QuantumCircuit(1, 1)   # exactly 1 qubit, starts in |0>
qc_main.h(0)
qc_main.t(0)
qc_main.h(0)
qc_main.measure(0, 0)

print('Gates before measurement: 3 (limit is 4)')
display(qc_main.draw('mpl'))

In [ ]:
counts_main = run(qc_main)
report('Main circuit (H, T, H)', counts_main)
print('Predicted: P(0) = %.3f, P(1) = %.3f' % (np.cos(np.pi/8)**2, np.sin(np.pi/8)**2))
plot_histogram(counts_main, title='Main circuit: H-T-H (100 shots)')
plt.show()

## 4. Bonus — two circuits, one favouring `0` and one favouring `1`

In [ ]:
# Circuit favouring 0:  H, T, H
qc_zero = QuantumCircuit(1, 1)
qc_zero.h(0); qc_zero.t(0); qc_zero.h(0)
qc_zero.measure(0, 0)

# Circuit favouring 1:  X, H, T, H   (4 gates, still within the limit)
qc_one = QuantumCircuit(1, 1)
qc_one.x(0); qc_one.h(0); qc_one.t(0); qc_one.h(0)
qc_one.measure(0, 0)

print('Circuit favouring 0:'); display(qc_zero.draw('mpl'))
print('Circuit favouring 1:'); display(qc_one.draw('mpl'))

In [ ]:
counts_zero = run(qc_zero)
counts_one = run(qc_one)
report('Favouring 0 (H,T,H)', counts_zero)
report('Favouring 1 (X,H,T,H)', counts_one)

plot_histogram([counts_zero, counts_one],
               legend=['0-biased: H,T,H', '1-biased: X,H,T,H'],
               title='Bonus: opposite biases (100 shots each)')
plt.show()

In [ ]:
labels = ['0', '1']
zero_bias = [counts_zero.get(k, 0) for k in labels]
one_bias = [counts_one.get(k, 0) for k in labels]

x = list(range(len(labels)))
w = 0.38
fig, ax = plt.subplots(figsize=(7, 4.5))
bars_z = ax.bar([i - w / 2 for i in x], zero_bias, w, label='0-biased (H, T, H)')
bars_o = ax.bar([i + w / 2 for i in x], one_bias, w, label='1-biased (X, H, T, H)')
ax.bar_label(bars_z)
ax.bar_label(bars_o)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlabel('Measured outcome')
ax.set_ylabel('Counts out of 100 shots')
ax.set_title('0-biased vs 1-biased circuit (100 shots each)')
ax.legend()
plt.show()

## 5. Different degrees of bias with `RY(θ)`
Only one gate is used, so the limit of 4 gates is respected. Prediction: P(1) = sin²(θ/2).

In [ ]:
angles = {'θ = π/6': np.pi/6, 'θ = π/3': np.pi/3, 'θ = π/2': np.pi/2,
          'θ = 2π/3': 2*np.pi/3, 'θ = π': np.pi}

ry_counts = []
for label, theta in angles.items():
    qc = QuantumCircuit(1, 1)
    qc.ry(theta, 0)
    qc.measure(0, 0)
    c = run(qc)
    ry_counts.append(c)
    report(f'RY({label})  [predicted P(1)={np.sin(theta/2)**2:.1%}]', c)

plot_histogram(ry_counts, legend=list(angles.keys()),
               title='Different degrees of bias with RY(θ) (100 shots each)', figsize=(11, 5))
plt.show()

## 6. Prediction vs. actual results

In [ ]:
rows = [('H,T,H', np.cos(np.pi/8)**2, counts_zero),
        ('X,H,T,H', np.sin(np.pi/8)**2, counts_one)]
print(f"{'Circuit':<10}{'Predicted P(0)':>16}{'Actual P(0)':>14}")
for name, p0, c in rows:
    print(f"{name:<10}{p0:>16.1%}{c.get('0', 0)/SHOTS:>14.1%}")

print()
print(f"{'RY angle':<12}{'Predicted P(1)':>16}{'Actual P(1)':>14}")
for (label, theta), c in zip(angles.items(), ry_counts):
    print(f"{label:<12}{np.sin(theta/2)**2:>16.1%}{c.get('1', 0)/SHOTS:>14.1%}")

**Short comparison (edit after running):** The `H,T,H` circuit gave ___ zeros and ___ ones, close to the predicted 85% / 15%. The `X,H,T,H` circuit gave the opposite result: ___ zeros and ___ ones. The `RY(θ)` experiments showed that larger θ moves probability from `0` to `1`. Small differences from the exact predictions come from the randomness of measuring only 100 shots.

**How the gates changed the distribution:** `H` puts the qubit in an equal superposition. `T` changes the phase of the |1⟩ part, and the second `H` converts that phase difference into unequal amplitudes, so the probabilities are no longer 50/50. `X` flips the starting state, which swaps the bias. `RY(θ)` directly sets the amplitudes to cos(θ/2) and sin(θ/2).

## Challenge Question
**How can quantum gates change the probability of obtaining 0 or 1 when a qubit is measured?**

A qubit's state is α|0⟩ + β|1⟩, and measurement gives `0` with probability |α|² and `1` with probability |β|². Quantum gates are unitary operations that change the amplitudes α and β. Gates such as `X` swap them, `H` spreads them into an equal superposition, and rotation gates such as `RY(θ)` set them to chosen values (cos(θ/2) and sin(θ/2)). Phase gates (`Z`, `S`, `T`) don't change the probabilities by themselves, but when followed by another `H` the phase difference interferes and changes the amplitudes. By choosing and ordering gates, we control the amplitudes and therefore the probabilities of measuring 0 or 1.